In [1]:
class Node:
    
    def __init__(self, state, parent = None, action = None, path_cost = 0):
        self.state = state          # o estado ao qual o nó corresponde; (str)
        self.parent = parent        # o nó da árvore que gerou este nó; (Node)
        self.action = action        # a ação executada para gerar este nó; (str)
        self.path_cost = path_cost  # o custo do caminho do nó inicial até este nó. (int)

In [2]:
class Frontier:
    def __init__(self):
        self.elements = []          # lista de nós
    def is_empty(self):
        # IS-EMPTY(frontier) returns true only if there are no nodes in the frontier.
        return len(self.elements) == 0
    def pop(self):
        # POP(frontier) removes the top node from the frontier and returns it.
        return self.elements.pop(0)
    def top(self):
        # TOP(frontier) returns (but does not remove) the top node of the frontier.
        return self.elements[0]
    def add(self, node):
        #ADD(node, frontier) inserts node into its proper place in the queue.
        pass

In [3]:
class PriorityQueue(Frontier):
        
    def __init__(self, f):
        super().__init__()            # lista de nós
        self.f = f                  # função de avaliação f
    
    def add(self, node):
        # ADD(node, frontier) inserts node into its proper place in the queue.
        self.elements.append(node)
        self.elements = sorted(self.elements, key = self.f)

In [4]:
class FIFOQueue(Frontier):
        
    def add(self, node):
        '''
        ADD(node, frontier) inserts node at the end of the queue.
        '''
        self.elements.append(node)

In [5]:
class LIFOQueue(Frontier):
            
        def add(self, node):
            '''
            ADD(node, frontier) inserts node at the beginning of the queue.
            '''
            self.elements.insert(0, node)

In [6]:
class PriorityQueueK(Frontier): 
        
    def __init__(self,k, f):
        super().__init__()            # lista de nós
        self.f = f                  # função de avaliação f
        self.k = k                  # número de nós a serem expandidos
    
    def add(self, node):
        # ADD(node, frontier) inserts node into its proper place in the queue.
        self.elements.append(node)
        self.elements = sorted(self.elements, key = self.f)
        if len(self.elements) > self.k:
            self.elements = self.elements[:self.k]

In [7]:
class Problem:

    def __init__(self, states, initial, goal, actions, transition_model, cost):
        self.states = states                #estados possíveis
        if initial not in states:           #verifica se o estado inicial é um estado possível
            self.states.append(initial)     #caso não seja, adiciona o estado inicial aos estados possíveis
        self.initial = initial              #estado inicial do problema
        if goal not in states:              #verifica se o estado objetivo é um estado possível
            self.states.append(goal)        #caso não seja, adiciona o estado objetivo aos estados possíveis
        self.goal = goal                    #estado(s) objetivo do problema
        self.actions = actions              #ações possíveis
        self.transition_model = transition_model
        self.cost = cost
    def get_actions(self, state):
        return self.actions[state]
    def result(self, state, action):
        return self.transition_model[state][action]
    def goal_test(self, state):
        return state == self.goal
    def action_cost(self, state1, action, state2):
        if action in self.actions[state1] and state2 == self.result(state1, action):
            return self.cost[state1][state2]
        else:
            return -1

In [8]:
'''
function BEST-FIRST-SEARCH(problem, f ) returns a solution node or failure 
    node←NODE(STATE=problem.INITIAL)
    frontier ← a priority queue ordered by f , with node as an element
    reached←a lookup table, with one entry with key problem.INITIAL and value node 
    while not IS-EMPTY(frontier) do
        node←POP(frontier)
        if problem.IS-GOAL(node.STATE) then return node 
        for each child in EXPAND(problem, node) do
            s←child.STATE
            if s is not in reached or child.PATH-COST < reached[s].PATH-COST then
                reached[s] ← child
                add child to frontier 
    return failure

function EXPAND(problem,node) yields nodes 
    s←node.STATE
    for each action in problem.ACTIONS(s) do
        s′ ←problem.RESULT(s,action)
        cost←node.PATH-COST + problem.ACTION-COST(s,action,s′)
        yield NODE(STATE=s′, PARENT=node, ACTION=action, PATH-COST=cost)
'''

class BestFirstSearch:
    def __init__(self, problem, f):
        self.problem = problem
        self.f = f
    def search(self, limit = 20):
        nExpanded = 0
        node = Node(self.problem.initial) # node←NODE(STATE=problem.INITIAL)
        frontier = PriorityQueue(self.f) # frontier ← a priority queue ordered by f , with node as an element
        frontier.add(node) # add node to frontier
        reached = {self.problem.initial: node} # reached←a lookup table, with one entry with key problem.INITIAL and value node
        while not frontier.is_empty(): # while not IS-EMPTY(frontier) do
            print("Frontier: ", end="")
            for node in frontier.elements:
                print(node.state, end=" ")
            print()
            node = frontier.pop() # node←POP(frontier)
            nExpanded += 1
            if nExpanded > limit:
                return None
            if self.problem.goal_test(node.state):
                return node
            else:
                print(f"{nExpanded} - {node.state} {node.path_cost} {self.f(node)}")
            for child in self.expand(node):
                s = child.state
                #if s not in reached or child.path_cost < reached[s].path_cost():
                if s not in reached or self.f(child) < self.f(reached[s]):
                    reached[s] = child
                    frontier.add(child)
        return None
    
    def expand(self, node):
        s = node.state
        for action in self.problem.get_actions(s):
            s_prime = self.problem.result(s, action)
            cost = node.path_cost + self.problem.action_cost(s, action, s_prime)
            yield Node(s_prime, node, action, cost)
    def path(self, node):
        path_back = []
        while node:
            path_back.append(node)
            node = node.parent
        return path_back[::-1]

![Arad to Bucharest](Arad2Bucarest.png "Arad to Bucarest road map").

In [9]:

states = ['Arad', 'Zerind', 'Oradea', 'Sibiu', 'Timisoara', 
        'Lugoj', 'Mehadia', 'Drobeta', 'Craiova', 'Rimnicu Vilcea', 
        'Fagaras', 'Pitesti', 'Bucharest', 'Giurgiu', 'Urziceni', 
        'Hirsova', 'Eforie', 'Vaslui', 'Iasi', 'Neamt']
initial = 'Arad'
goal = 'Bucharest'
actions = {'Arad': ['toZerind', 'toSibiu', 'toTimisoara'],
        'Zerind': ['toArad', 'toOradea'],
        'Oradea': ['toZerind', 'toSibiu'],
        'Sibiu': ['toArad', 'toOradea', 'toFagaras', 'toRimnicu Vilcea'],
        'Timisoara': ['toArad', 'toLugoj'],
        'Lugoj': ['toTimisoara', 'toMehadia'],
        'Mehadia': ['toLugoj', 'toDrobeta'],
        'Drobeta': ['toMehadia', 'toCraiova'],
        'Craiova': ['toDrobeta', 'toRimnicu Vilcea', 'toPitesti'],
        'Rimnicu Vilcea': ['toSibiu', 'toCraiova', 'toPitesti'],
        'Fagaras': ['toSibiu', 'toBucharest'],
        'Pitesti': ['toRimnicu Vilcea', 'toCraiova', 'toBucharest'],
        'Bucharest': ['toFagaras', 'toPitesti', 'toGiurgiu', 'toUrziceni'],
        'Giurgiu': ['toBucharest'],
        'Urziceni': ['toBucharest', 'toHirsova', 'toVaslui'],
        'Hirsova': ['toUrziceni', 'toEforie'],
        'Eforie': ['toHirsova'],
        'Vaslui': ['toUrziceni', 'toIasi'],
        'Iasi': ['toVaslui', 'toNeamt'],
        'Neamt': ['toIasi']}
transition_model = {
    'Arad': {'toZerind': 'Zerind', 'toSibiu': 'Sibiu', 'toTimisoara': 'Timisoara'},
    'Zerind': {'toArad': 'Arad', 'toOradea': 'Oradea'},
    'Oradea': {'toZerind': 'Zerind', 'toSibiu': 'Sibiu'},
    'Sibiu': {'toArad': 'Arad', 'toOradea': 'Oradea', 'toFagaras': 'Fagaras', 'toRimnicu Vilcea': 'Rimnicu Vilcea'},
    'Timisoara': {'toArad': 'Arad', 'toLugoj': 'Lugoj'},
    'Lugoj': {'toTimisoara': 'Timisoara', 'toMehadia': 'Mehadia'},
    'Mehadia': {'toLugoj': 'Lugoj', 'toDrobeta': 'Drobeta'},
    'Drobeta': {'toMehadia': 'Mehadia', 'toCraiova': 'Craiova'},
    'Craiova': {'toDrobeta': 'Drobeta', 'toRimnicu Vilcea': 'Rimnicu Vilcea', 'toPitesti': 'Pitesti'},
    'Rimnicu Vilcea': {'toSibiu': 'Sibiu', 'toCraiova': 'Craiova', 'toPitesti': 'Pitesti'},
    'Fagaras': {'toSibiu': 'Sibiu', 'toBucharest': 'Bucharest'},
    'Pitesti': {'toRimnicu Vilcea': 'Rimnicu Vilcea', 'toCraiova': 'Craiova', 'toBucharest': 'Bucharest'},
    'Bucharest': {'toFagaras': 'Fagaras', 'toPitesti': 'Pitesti', 'toGiurgiu': 'Giurgiu', 'toUrziceni': 'Urziceni'},
    'Giurgiu': {'toBucharest': 'Bucharest'},
    'Urziceni':{'toBucharest': 'Bucharest', 'toHirsova': 'Hirsova', 'toVaslui': 'Vaslui'},
    'Hirsova': {'toUrziceni': 'Urziceni', 'toEforie': 'Eforie'},
    'Eforie': {'toHirsova': 'Hirsova'},
    'Vaslui': {'toUrziceni': 'Urziceni', 'toIasi': 'Iasi'},
    'Iasi': {'toVaslui': 'Vaslui', 'toNeamt': 'Neamt'},
    'Neamt': {'toIasi': 'Iasi'}}
cost = {'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
        'Zerind': {'Arad': 75, 'Oradea': 71},
        'Oradea': {'Zerind': 71, 'Sibiu': 151},
        'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
        'Timisoara': {'Arad': 118, 'Lugoj': 111},
        'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
        'Mehadia': {'Lugoj': 70, 'Drobeta': 75},
        'Drobeta': {'Mehadia': 75, 'Craiova': 120},
        'Craiova': {'Drobeta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
        'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
        'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
        'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucharest': 101},
        'Bucharest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
        'Giurgiu': {'Bucharest': 90},
        'Urziceni': {'Bucharest': 85, 'Hirsova': 98, 'Vaslui': 142},
        'Hirsova': {'Urziceni': 98, 'Eforie': 86},
        'Eforie': {'Hirsova': 86},
        'Vaslui': {'Urziceni': 142, 'Iasi': 92},
        'Iasi': {'Vaslui': 92, 'Neamt': 87},
        'Neamt': {'Iasi': 87}}
Arad2Bucarest = Problem(states, initial, goal, actions, transition_model, cost)

# Uninformed Search Strategies

## Breadth-first search

* This is a systematic search strategy that is therefore complete even on infinite state spaces.

* We could implement breadth-first search as a call to BEST-FIRST-SEARCH where the evaluation function $f(n)$ is the depth of the node—that is, the number of actions it takes to reach the node.

In [10]:
def depth(node):
    node_depth = 0
    while node:
        node_depth += 1
        node = node.parent
    return node_depth

busca = BestFirstSearch(Arad2Bucarest, depth)
print("Breadth-first search (Arad -> Bucharest):")
node = busca.search()
print(node.state, node.path_cost)
print("__________________________")
print("Solution:")
for step in busca.path(node):
    print(step.state, step.path_cost)

Breadth-first search (Arad -> Bucharest):
Frontier: Arad 
1 - Arad 0 1
Frontier: Zerind Sibiu Timisoara 
2 - Zerind 75 2
Frontier: Sibiu Timisoara Oradea 
3 - Sibiu 140 2
Frontier: Timisoara Oradea Fagaras Rimnicu Vilcea 
4 - Timisoara 118 2
Frontier: Oradea Fagaras Rimnicu Vilcea Lugoj 
5 - Oradea 146 3
Frontier: Fagaras Rimnicu Vilcea Lugoj 
6 - Fagaras 239 3
Frontier: Rimnicu Vilcea Lugoj Bucharest 
7 - Rimnicu Vilcea 220 3
Frontier: Lugoj Bucharest Craiova Pitesti 
8 - Lugoj 229 3
Frontier: Bucharest Craiova Pitesti Mehadia 
Bucharest 450
__________________________
Solution:
Arad 0
Sibiu 140
Fagaras 239
Bucharest 450


![Breadth-first search](Breadth-first_search.png "Breadth-first search tree")

However, we can get additional efficiency with a couple of tricks. 
* A first-in-first-out queue will be faster than a priority queue, and will give us the correct order of nodes: new nodes (which are always deeper than their parents) go to the back of the queue, and old nodes, which are shallower than the new nodes, get expanded first. 
* Reached can be a set of states rather than a mapping from states to nodes, because once we’ve reached a state, we can never find a better path to the state. That also means we can do an early goal test, checking whether a node is a solution as soon as it is generated, rather than the late goal test that best-first search uses, waiting until a node is popped off the queue. 

In [11]:
'''
function BREADTH-FIRST-SEARCH(problem) returns a solution node or failure 
    node←NODE(problem.INITIAL)
    if problem.IS-GOAL(node.STATE) then return node
    frontier ← a FIFO queue, with node as an element 
    reached←{problem.INITIAL}
    while not IS-EMPTY(frontier) do 
        node←POP(frontier)
        for each child in EXPAND(problem, node) do
            s←child.STATE
            if problem.IS-GOAL(s) then return child 
            if s is not in reached then
                add s to reached
                add child to frontier 
    return failure

function UNIFORM-COST-SEARCH(problem) returns a solution node, or failure 
    return BEST-FIRST-SEARCH(problem, PATH-COST)
'''

class BreadthFirstSearch:
    
    def __init__(self, problem):
        self.problem = problem
    
    def search(self):
        nExpanded = 1
        node = Node(self.problem.initial)
        if self.problem.goal_test(node.state):
            return node
        else:
            print(f"{nExpanded} - {node.state} {node.path_cost} {self.f(node)}")
        frontier = FIFOQueue()
        frontier.add(node)
        reached = [self.problem.initial]  # reached←{problem.INITIAL} (como uma lista)
        while not frontier.is_empty():
            print("Frontier: ", end="")
            for node in frontier.elements:
                print(node.state, end=" ")
            print()
            node = frontier.pop()
            for child in self.expand(node):
                s = child.state
                nExpanded += 1
                if self.problem.goal_test(s):
                    return child
                else:
                    print(f"{nExpanded} - {child.state} {child.path_cost} {self.f(child)}")
                if s not in reached:
                    reached.append(s)
                    frontier.add(child)
        return None
    
    def f(self, node):
        node_depth = 0
        while node:
            node_depth += 1
            node = node.parent
        return node_depth

    def expand(self, node):
        s = node.state
        for action in self.problem.get_actions(s):
            s_prime = self.problem.result(s, action)
            cost = node.path_cost + self.problem.action_cost(s, action, s_prime)
            yield Node(s_prime, node, action, cost)
    def path(self, node):
        path_back = []
        while node:
            path_back.append(node)
            node = node.parent
        return path_back[::-1]

In [12]:
print("__________________________")
print("Breadth-First Search (Arad -> Bucharest):")
busca = BreadthFirstSearch(Arad2Bucarest)
node = busca.search()
print(node.state, node.path_cost)
print("__________________________")
print("Solution:")
for step in busca.path(node):
    print(step.state, step.path_cost)

__________________________
Breadth-First Search (Arad -> Bucharest):
1 - Arad 0 1
Frontier: Arad 
2 - Zerind 75 2
3 - Sibiu 140 2
4 - Timisoara 118 2
Frontier: Zerind Sibiu Timisoara 
5 - Arad 150 3
6 - Oradea 146 3
Frontier: Sibiu Timisoara Oradea 
7 - Arad 280 3
8 - Oradea 291 3
9 - Fagaras 239 3
10 - Rimnicu Vilcea 220 3
Frontier: Timisoara Oradea Fagaras Rimnicu Vilcea 
11 - Arad 236 3
12 - Lugoj 229 3
Frontier: Oradea Fagaras Rimnicu Vilcea Lugoj 
13 - Zerind 217 4
14 - Sibiu 297 4
Frontier: Fagaras Rimnicu Vilcea Lugoj 
15 - Sibiu 338 4
Bucharest 450
__________________________
Solution:
Arad 0
Sibiu 140
Fagaras 239
Bucharest 450


![Breadth-first search](Breadth-first_search_FIFOQueue.png "Breadth-first search tree").

## Dijkstra’s algorithm or uniform-cost search

In [13]:

print("__________________________")
print("Breadth-First Search (Arad -> Bucharest):")
busca = BestFirstSearch(Arad2Bucarest, lambda node: node.path_cost)
node = busca.search()
print(node.state, node.path_cost)
print("__________________________")
print("Solution:")
for step in busca.path(node):
    print(step.state, step.path_cost)

__________________________
Breadth-First Search (Arad -> Bucharest):
Frontier: Arad 
1 - Arad 0 0
Frontier: Zerind Timisoara Sibiu 
2 - Zerind 75 75
Frontier: Timisoara Sibiu Oradea 
3 - Timisoara 118 118
Frontier: Sibiu Oradea Lugoj 
4 - Sibiu 140 140
Frontier: Oradea Rimnicu Vilcea Lugoj Fagaras 
5 - Oradea 146 146
Frontier: Rimnicu Vilcea Lugoj Fagaras 
6 - Rimnicu Vilcea 220 220
Frontier: Lugoj Fagaras Pitesti Craiova 
7 - Lugoj 229 229
Frontier: Fagaras Mehadia Pitesti Craiova 
8 - Fagaras 239 239
Frontier: Mehadia Pitesti Craiova Bucharest 
9 - Mehadia 299 299
Frontier: Pitesti Craiova Drobeta Bucharest 
10 - Pitesti 317 317
Frontier: Craiova Drobeta Bucharest Bucharest 
11 - Craiova 366 366
Frontier: Drobeta Bucharest Bucharest 
12 - Drobeta 374 374
Frontier: Bucharest Bucharest 
Bucharest 418
__________________________
Solution:
Arad 0
Sibiu 140
Rimnicu Vilcea 220
Pitesti 317
Bucharest 418


![Breadth-first search](Uniform-cost_search.png "Uniform cost search tree").

## 

## Depth-first search and the problem of memory

In [14]:
def negdepth(node):
    node_depth = 0
    while node:
        node_depth += 1
        node = node.parent
    return -1*node_depth

busca = BestFirstSearch(Arad2Bucarest, negdepth)
print("Depth-first search (Arad -> Bucharest):")
node = busca.search(limit=100)
if node:
    print(node.state, node.path_cost)
    print("__________________________")
    print("Solution:")
    for step in busca.path(node):
        print(step.state, step.path_cost)
else:
    print("Solution not found")


Depth-first search (Arad -> Bucharest):
Frontier: Arad 
1 - Arad 0 -1
Frontier: Zerind Sibiu Timisoara 
2 - Zerind 75 -2
Frontier: Arad Oradea Sibiu Timisoara 
3 - Arad 150 -3
Frontier: Zerind Sibiu Timisoara Oradea Sibiu Timisoara 
4 - Zerind 225 -4
Frontier: Arad Oradea Sibiu Timisoara Oradea Sibiu Timisoara 
5 - Arad 300 -5
Frontier: Zerind Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara 
6 - Zerind 375 -6
Frontier: Arad Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara 
7 - Arad 450 -7
Frontier: Zerind Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara 
8 - Zerind 525 -8
Frontier: Arad Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara 
9 - Arad 600 -9
Frontier: Zerind Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoara 
10 - Zerind 675 -10
Frontier: Arad Oradea Sibiu Timisoara Oradea Sibiu Timisoara Oradea Sibiu Timisoa

![Depth-first search](Depth-first_search.png "Depth first search tree").

## Depth-limited search

In [16]:
""" 
function DEPTH-LIMITED-SEARCH(problem, l) returns a node or failure or cutoff 
    frontier←a LIFO queue (stack) with NODE(problem.INITIAL) as an element 
    result ← failure
    while not IS-EMPTY(frontier) do
        node←POP(frontier)
        if problem.IS-GOAL(node.STATE) then return node 
        if DEPTH(node) > l then
            result ← cutoff
        else if not IS-CYCLE(node) do
            for each child in EXPAND(problem, node) do 
                add child to frontier
    return result
"""

class DepthLimitedSearch:

    def __init__(self, problem):
        self.problem = problem

    def search(self, l=5):
        node = Node(self.problem.initial)
        frontier = LIFOQueue()
        frontier.add(node)
        result = None
        nExpanded = 0
        while not frontier.is_empty():
            node = frontier.pop()
            if self.problem.goal_test(node.state):
                return node
            else:
                nExpanded += 1
                print(f"{nExpanded} - {node.state} {node.path_cost} {self.depth(node)}")
            if self.depth(node) > l:
                result = 'cutoff'
            elif not self.is_cycle(node):
                for child in self.expand(node):
                    frontier.add(child)
        return result

    def expand(self, node):
        s = node.state
        for action in self.problem.get_actions(s):
            s_prime = self.problem.result(s, action)
            cost = node.path_cost + self.problem.action_cost(s, action, s_prime)
            yield Node(s_prime, node, action, cost)

    def depth(self, node):
        depth = 0
        while node:
            depth += 1
            node = node.parent
        return depth

    def is_cycle(self, node):
        state = node.state
        while node:
            node = node.parent
            if node and node.state == state:
                return True
        return False
    
    def path(self, node):
        path_back = []
        while node:
            path_back.append(node)
            node = node.parent
        return path_back[::-1]

In [17]:
busca = DepthLimitedSearch(Arad2Bucarest)
print("Depth Limited search N = 3(Arad -> Bucharest):")
node = busca.search(2)
if node and isinstance(node, Node):
    print(node.state, node.path_cost)
    print("__________________________")
    print("Solution:")
    for step in busca.path(node):
        print(step.state, step.path_cost)
elif node and node == 'cutoff':
    print("Solution not found with this depth limit")
else:
    print("Solution not found")

Depth Limited search N = 3(Arad -> Bucharest):
1 - Arad 0 1
2 - Timisoara 118 2
3 - Lugoj 229 3
4 - Arad 236 3
5 - Sibiu 140 2
6 - Rimnicu Vilcea 220 3
7 - Fagaras 239 3
8 - Oradea 291 3
9 - Arad 280 3
10 - Zerind 75 2
11 - Oradea 146 3
12 - Arad 150 3
Solution not found with this depth limit


![Depth-limited search N=3.png](Depth-limited_search_N3.png "Depth-limited N = 3 search tree").

In [18]:
print("Depth Limited search N = 3(Arad -> Bucharest):")
node = busca.search(3)
if node and isinstance(node, Node):
    print(node.state, node.path_cost)
    print("__________________________")
    print("Solution:")
    for step in busca.path(node):
        print(step.state, step.path_cost)
elif node and node == 'cutoff':
    print("Solution not found with this depth limit")
else:
    print("Solution not found")

Depth Limited search N = 3(Arad -> Bucharest):
1 - Arad 0 1
2 - Timisoara 118 2
3 - Lugoj 229 3
4 - Mehadia 299 4
5 - Timisoara 340 4
6 - Arad 236 3
7 - Sibiu 140 2
8 - Rimnicu Vilcea 220 3
9 - Pitesti 317 4
10 - Craiova 366 4
11 - Sibiu 300 4
12 - Fagaras 239 3
Bucharest 450
__________________________
Solution:
Arad 0
Sibiu 140
Fagaras 239
Bucharest 450


![Depth-limited search N=4.png](Depth-limited_search_N4.png "Depth-limited N = 4 search tree").

In [19]:
print("Depth Limited search N = 4(Arad -> Bucharest):")
node = busca.search(4)
if node and isinstance(node, Node):
    print(node.state, node.path_cost)
    print("__________________________")
    print("Solution:")
    for step in busca.path(node):
        print(step.state, step.path_cost)
elif node and node == 'cutoff':
    print("Solution not found with this depth limit")
else:
    print("Solution not found")

Depth Limited search N = 4(Arad -> Bucharest):
1 - Arad 0 1
2 - Timisoara 118 2
3 - Lugoj 229 3
4 - Mehadia 299 4
5 - Drobeta 374 5
6 - Lugoj 369 5
7 - Timisoara 340 4
8 - Arad 236 3
9 - Sibiu 140 2
10 - Rimnicu Vilcea 220 3
11 - Pitesti 317 4
Bucharest 418
__________________________
Solution:
Arad 0
Sibiu 140
Rimnicu Vilcea 220
Pitesti 317
Bucharest 418


![Depth-limited search N=5.png](Depth-limited_search_N5.png "Depth-limited N = 5 search tree").

## Iterative deepening search

In [20]:
""" 
function ITERATIVE-DEEPENING-SEARCH(problem) returns a solution node or failure 
    for depth = 0 to ∞ do
    result←DEPTH-LIMITED-SEARCH(problem,depth) 
    if result ̸= cutoff then return result
"""

class IterativeDeepeningSearch:

    def __init__(self, problem):
        self.problem = problem
        self.depth_limited_search = DepthLimitedSearch(problem)

    def search(self):
        for depth in range(0, 10000):
            print()
            print(f"Depth: {depth}")
            print("__________________________")
            result = self.depth_limited_search.search(depth)
            if result != 'cutoff':
                return result
        return None
    
    def path(self, node):
        path_back = []
        while node:
            path_back.append(node)
            node = node.parent
        return path_back[::-1]

In [21]:
busca = IterativeDeepeningSearch(Arad2Bucarest)
print("Iterative deepening search (Arad -> Bucharest):")
node = busca.search()
if node and isinstance(node, Node):
    print(node.state, node.path_cost)
    print("__________________________")
    print("Solution:")
    for step in busca.path(node):
        print(step.state, step.path_cost)
elif node and node == 'cutoff':
    print("Solution not found with this depth limit")
else:
    print("Solution not found")

Iterative deepening search (Arad -> Bucharest):

Depth: 0
__________________________
1 - Arad 0 1

Depth: 1
__________________________
1 - Arad 0 1
2 - Timisoara 118 2
3 - Sibiu 140 2
4 - Zerind 75 2

Depth: 2
__________________________
1 - Arad 0 1
2 - Timisoara 118 2
3 - Lugoj 229 3
4 - Arad 236 3
5 - Sibiu 140 2
6 - Rimnicu Vilcea 220 3
7 - Fagaras 239 3
8 - Oradea 291 3
9 - Arad 280 3
10 - Zerind 75 2
11 - Oradea 146 3
12 - Arad 150 3

Depth: 3
__________________________
1 - Arad 0 1
2 - Timisoara 118 2
3 - Lugoj 229 3
4 - Mehadia 299 4
5 - Timisoara 340 4
6 - Arad 236 3
7 - Sibiu 140 2
8 - Rimnicu Vilcea 220 3
9 - Pitesti 317 4
10 - Craiova 366 4
11 - Sibiu 300 4
12 - Fagaras 239 3
Bucharest 450
__________________________
Solution:
Arad 0
Sibiu 140
Fagaras 239
Bucharest 450
